# Vizabridge 비자 RAG 파이프라인 — 단계별 시각화

이 노트북은 **9단계 데이터 전처리 파이프라인**의 각 단계를 시각화하고 검증합니다.

```
data/raw/*.hwp
   │  Stage 1  scripts/parse_hwp_to_markdown.py             (Python, kordoc subprocess)
   ▼
data/parsed/raw/{stay,visa}_manual.md
   │  Stage 2  scripts/index_markdown_chunks.py             (Python, deterministic)
   ▼
data/parsed/chunks/{stay,visa}_chunks_index.jsonl
   │  Stage 3  /vizabridge-normalize                        (Claude Code skill)
   ▼
data/parsed/normalized/{stay,visa}_manual.md
   │  Stage 4  scripts/validate_normalization.py            (Python, deterministic)
   │  Stage 5  /vizabridge-repair (조건부)                   (Claude Code skill)
   │  Stage 6  scripts/build_semantic_csv.py                (Python, deterministic)
   ▼
data/processed/{stay,visa}_manual_semantic_clean.csv
   │  Stage 7  /vizabridge-enrich-chatbot                   (Claude Code skill)
   ▼
data/parsed/normalized_chatbot/{stay,visa}_manual.md
   │  Stage 8  scripts/build_chatbot_csv.py                 (Python, deterministic)
   ▼
data/processed/{stay,visa}_manual_chatbot_ready.csv
   │  Stage 9  scripts/quality_report_semantic_manual_csvs.py
   ▼
output/quality/*, output/review/*
```

LLM은 stages 3, 5, 7에서만 사용합니다. 나머지 6단계는 모두 결정적 Python.

## 이 노트북에서 검증하는 것
- 각 단계 산출물이 정상 생성되었는지
- 각 단계 CSV/JSONL의 컬럼 구조, 행 수, 누락률
- Pandas 테이블 미리보기
- Plotly 시각화 (분포, 누락률 히트맵, 비자코드 그래프)
- 단계 간 데이터가 일관되게 흐르는지 (entity preservation)


## Setup

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJ = Path.cwd()
while not (PROJ / "scripts" / "parse_hwp_to_markdown.py").exists():
    PROJ = PROJ.parent
print(f"PROJECT_ROOT: {PROJ}")

RAW = PROJ / "data" / "parsed" / "raw"
CHUNKS = PROJ / "data" / "parsed" / "chunks"
NORM = PROJ / "data" / "parsed" / "normalized"
NORM_CB = PROJ / "data" / "parsed" / "normalized_chatbot"
VAL = PROJ / "data" / "parsed" / "validation"
PROC = PROJ / "data" / "processed"
QUAL = PROJ / "output" / "quality"
REV = PROJ / "output" / "review"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


PROJECT_ROOT: /Users/junghwan/Desktop/Vizabridge/.claude/worktrees/great-bardeen-a27f1a


## Stage 1: HWP → Markdown (kordoc)

원본 HWP를 [kordoc](https://github.com/chrisryugj/kordoc)으로 Markdown으로 변환합니다.
표는 HTML `<table>` 블록으로 보존되며, 한글이 깨지지 않습니다.

```bash
python scripts/parse_hwp_to_markdown.py
```

`data/raw/*.hwp` → `data/parsed/raw/{stay,visa}_manual.md`


In [2]:
# 원본 HWP와 변환된 MD의 메타데이터
def file_meta(p: Path):
    return {
        "file": p.name,
        "size_KB": p.stat().st_size // 1024,
        "exists": p.exists(),
    }

raw_hwps = sorted((PROJ / "data" / "raw").glob("*.hwp"))
parsed_mds = sorted(RAW.glob("*.md"))

meta_df = pd.DataFrame([file_meta(p) for p in raw_hwps + parsed_mds])
meta_df["kind"] = ["HWP"] * len(raw_hwps) + ["Markdown"] * len(parsed_mds)
display(meta_df)


,file,size_KB,exists,kind
0,260504 사증민원 자격별 안내 매뉴얼.hwp,2102,True,HWP
1,260504 체류민원 자격별 안내 매뉴얼.hwp,3122,True,HWP
2,stay_manual.md,1612,True,Markdown
3,visa_manual.md,1059,True,Markdown


In [3]:
# stay_manual.md 첫 100줄 미리보기 (kordoc 출력 구조 확인)
preview = (RAW / "stay_manual.md").read_text(encoding="utf-8").splitlines()
print(f"총 {len(preview)}줄, {sum(len(l) for l in preview)//1024} KB")
print("\n=== 첫 50줄 ===")
print("\n".join(preview[:50]))


총 15952줄, 783 KB

=== 첫 50줄 ===
## 외국인체류 안내매뉴얼


# 2026. 5.

법무부
출입국․외국인정책본부

<table>
<tr><th colspan="2">目 次</th></tr>
<tr><td colspan="2"></td></tr>
<tr><td>‣ 각종 체류허가 신청 시 유의사항</td><td>‣ 공통사항 (체류 일반)</td></tr>
<tr><td colspan="2">체류자격별 대상 및 제출서류 등 안내메뉴얼</td></tr>
<tr><td>1. 외교(A-1)</td><td>20. 회화지도(E-2)</td></tr>
<tr><td>2. 공무(A-2)</td><td>21. 연구(E-3)</td></tr>
<tr><td>3. 협정(A-3)</td><td>22. 기술지도(E-4)</td></tr>
<tr><td>4. 사증면제(B-1)</td><td>23. 전문직업(E-5)</td></tr>
<tr><td>5. 관광통과(B-2)</td><td>24. 예술흥행(E-6)</td></tr>
<tr><td>6. 일시취재(C-1)</td><td>25. 특정활동(E-7)</td></tr>
<tr><td>7. 단기방문(C-3)</td><td>26. 계절근로(E-8)</td></tr>
<tr><td>8. 단기취업(C-4)</td><td>27. 비전문취업(E-9)</td></tr>
<tr><td>9. 문화예술(D-1)</td><td>28. 선원취업(E-10)</td></tr>
<tr><td>10. 유학(D-2)</td><td>29. 방문동거(F-1)</td></tr>
<tr><td>11. 기술연수(D-3)</td><td>30. 거주(F-2)</td></tr>
<tr><td>12. 일반연수(D-4)</td><td>31. 동반(F-3)</td></tr>
<tr><td>13. 취재(D-5)</td><td>32. 영주(F-5):동포,난민제외</td></tr>
<tr><td>14. 종교(D-6)</td><td>33. 결혼이민(F-6)</td></t

## Stage 2: Markdown → 청크 인덱스

`<table>` 경계 + 누적 크기 기준으로 결정적 분할. 각 청크는 그 안에서 발견되는 모든 비자코드를 메타데이터로 부착합니다.

```bash
python scripts/index_markdown_chunks.py
```


In [4]:
def load_chunks(manual_key: str) -> pd.DataFrame:
    rows = []
    with (CHUNKS / f"{manual_key}_chunks_index.jsonl").open(encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    df["num_codes"] = df["visa_codes"].map(len)
    return df

stay_chunks = load_chunks("stay")
visa_chunks = load_chunks("visa")
print(f"stay: {len(stay_chunks)} chunks")
print(f"visa: {len(visa_chunks)} chunks")

display(stay_chunks[["chunk_id", "start_line", "end_line", "char_count", "table_count", "num_codes", "oversized"]].head(10))


stay: 34 chunks
visa: 23 chunks


,chunk_id,start_line,end_line,char_count,table_count,num_codes,oversized
0,stay_001,1,344,19282,4,45,False
1,stay_002,345,431,16211,8,47,False
2,stay_003,432,448,15341,1,40,False
3,stay_004,449,1185,33963,9,7,True
4,stay_005,1186,1307,21459,5,24,False
5,stay_006,1308,1558,20013,5,25,False
6,stay_007,1559,1664,23670,3,29,False
7,stay_008,1665,2047,27842,5,32,False
8,stay_009,2048,2214,25573,5,32,False
9,stay_010,2215,2246,16340,2,23,False


In [5]:
# 청크 크기 분포
fig = make_subplots(rows=1, cols=2, subplot_titles=("Stay 청크 크기 분포", "Visa 청크 크기 분포"))
fig.add_trace(go.Histogram(x=stay_chunks["char_count"], nbinsx=15, name="stay", marker_color="#2563eb"), row=1, col=1)
fig.add_trace(go.Histogram(x=visa_chunks["char_count"], nbinsx=15, name="visa", marker_color="#7c3aed"), row=1, col=2)
fig.add_vline(x=15000, line_dash="dash", line_color="green", annotation_text="target 15K", row=1, col=1)
fig.add_vline(x=15000, line_dash="dash", line_color="green", annotation_text="target 15K", row=1, col=2)
fig.add_vline(x=30000, line_dash="dash", line_color="red", annotation_text="oversized 30K", row=1, col=1)
fig.add_vline(x=30000, line_dash="dash", line_color="red", annotation_text="oversized 30K", row=1, col=2)
fig.update_layout(showlegend=False, height=400, title_text="청크 크기 분포 — 목표 ~15K chars, 30K 초과 시 sub-chunking 필요")
fig.show()


In [6]:
# 청크당 비자코드 수 + oversized 비율
fig = make_subplots(rows=1, cols=2, subplot_titles=("청크당 비자코드 수", "오버사이즈 청크 비율"))
fig.add_trace(go.Box(y=stay_chunks["num_codes"], name="stay", marker_color="#2563eb"), row=1, col=1)
fig.add_trace(go.Box(y=visa_chunks["num_codes"], name="visa", marker_color="#7c3aed"), row=1, col=1)

oversize_data = pd.DataFrame({
    "manual": ["stay", "stay", "visa", "visa"],
    "category": ["regular", "oversized", "regular", "oversized"],
    "count": [
        len(stay_chunks) - stay_chunks["oversized"].sum(),
        stay_chunks["oversized"].sum(),
        len(visa_chunks) - visa_chunks["oversized"].sum(),
        visa_chunks["oversized"].sum(),
    ],
})
for manual, color in [("stay", "#2563eb"), ("visa", "#7c3aed")]:
    sub = oversize_data[oversize_data["manual"] == manual]
    fig.add_trace(go.Bar(x=sub["category"], y=sub["count"], name=manual, marker_color=color, showlegend=False), row=1, col=2)
fig.update_layout(height=400, title_text="청크 메타데이터 통계")
fig.show()


In [7]:
# 발견된 비자코드 전체 목록 (sub-code 포함)
all_codes_stay = sorted(set().union(*stay_chunks["visa_codes"].tolist()))
all_codes_visa = sorted(set().union(*visa_chunks["visa_codes"].tolist()))
print(f"Stay에서 발견된 코드 수: {len(all_codes_stay)} (sub-code 포함)")
print(f"  → 메인 코드 종류: {sum(1 for c in all_codes_stay if c.count('-') == 1)}")
print(f"  → sub-code: {sum(1 for c in all_codes_stay if c.count('-') >= 2)}")
print()
print(f"Visa에서 발견된 코드 수: {len(all_codes_visa)} (sub-code 포함)")
print(f"  → 메인 코드 종류: {sum(1 for c in all_codes_visa if c.count('-') == 1)}")
print(f"  → sub-code: {sum(1 for c in all_codes_visa if c.count('-') >= 2)}")


Stay에서 발견된 코드 수: 181 (sub-code 포함)
  → 메인 코드 종류: 38
  → sub-code: 143

Visa에서 발견된 코드 수: 159 (sub-code 포함)
  → 메인 코드 종류: 38
  → sub-code: 121


## Stage 3: 청크 → 정규화 MD (Claude Code skill)

`/vizabridge-normalize` 스킬이 각 청크의 원본 Markdown을 읽어 행 단위 정규화 MD에 append합니다.
각 행은 (visa_code, petition_type, subsection_type) 조합 단위로 추출됩니다.

이 단계는 LLM 단계입니다. 결과물은 `data/parsed/normalized/{stay,visa}_manual.md`에 마커로 청크별 그룹화되어 저장됩니다.


In [8]:
# 정규화 MD 진행률
OPEN_RE = re.compile(r"<!--\s*vizabridge-normalize v1 chunk:\s*(\S+)\s+hash:\s*(\S+)\s+lines:\s*(\d+)-(\d+)\s*-->")
CLOSE_RE = re.compile(r"<!--\s*end chunk:\s*(\S+)\s*-->")
ROW_RE = re.compile(r"^###\s+row\s+", re.MULTILINE)

def normalize_progress(manual_key: str):
    norm_path = NORM / f"{manual_key}_manual.md"
    if not norm_path.exists():
        return {"manual": manual_key, "completed_chunks": 0, "total_rows": 0}
    text = norm_path.read_text(encoding="utf-8")
    opens = OPEN_RE.findall(text)
    closes = {m for m in CLOSE_RE.findall(text)}
    completed = [(cid, h, int(s), int(e)) for cid, h, s, e in opens if cid in closes]

    # Rows per chunk
    chunks_with_rows = []
    for open_m in OPEN_RE.finditer(text):
        cid = open_m.group(1)
        close_m = next((m for m in CLOSE_RE.finditer(text) if m.group(1) == cid and m.start() > open_m.end()), None)
        if close_m:
            body = text[open_m.end():close_m.start()]
            row_count = len(ROW_RE.findall(body))
            chunks_with_rows.append({"chunk_id": cid, "rows": row_count})
    chunks_df = pd.DataFrame(chunks_with_rows)
    total_rows = chunks_df["rows"].sum() if not chunks_df.empty else 0
    return {
        "manual": manual_key,
        "completed_chunks": len(completed),
        "total_rows": int(total_rows),
        "chunks_df": chunks_df,
    }

stay_p = normalize_progress("stay")
visa_p = normalize_progress("visa")
print(f"stay: {stay_p['completed_chunks']} chunks normalized, {stay_p['total_rows']} rows")
print(f"visa: {visa_p['completed_chunks']} chunks normalized, {visa_p['total_rows']} rows")


stay: 5 chunks normalized, 38 rows
visa: 1 chunks normalized, 6 rows


In [9]:
# 청크별 row 수 차트
fig = make_subplots(rows=1, cols=2, subplot_titles=("Stay: 청크별 row 수", "Visa: 청크별 row 수"))
for i, (p, color) in enumerate([(stay_p, "#2563eb"), (visa_p, "#7c3aed")], start=1):
    if "chunks_df" in p and not p["chunks_df"].empty:
        fig.add_trace(go.Bar(x=p["chunks_df"]["chunk_id"], y=p["chunks_df"]["rows"], marker_color=color), row=1, col=i)
fig.update_layout(showlegend=False, height=350, title_text="청크별 추출된 행 수")
fig.show()


In [10]:
# 정규화 MD 샘플 (stay_001의 row 1개)
norm_text = (NORM / "stay_manual.md").read_text(encoding="utf-8")
sample_match = re.search(r"### row .*?(?=<!--|### row)", norm_text, re.DOTALL)
if sample_match:
    sample = sample_match.group(0).strip()
    print("=== 정규화 MD 샘플 (첫 row) ===")
    print(sample[:2000] + ("..." if len(sample) > 2000 else ""))


=== 정규화 MD 샘플 (첫 row) ===
### row 공통사항 / 공통사항 / 수수료
- manual_type: 체류민원
- stay_status_code: 공통
- stay_status_name_ko: 공통사항
- item_type: common_rule
- section_title: 공통사항: 각종 체류허가 등에 관한 심사수수료
- subtype_or_program:
- petition_type: 공통사항
- subsection_type: 수수료
- applicant_context:
- eligibility:
- target_persons:
- common_documents:
- mandatory_documents:
- other_documents:
- requirements:
- procedure:
- restrictions:
- exceptions:
- fees: |
    체류자격 외 활동허가: 12만원
    근무처의 변경·추가: 12만원
    체류자격 부여: 8만원
    체류자격 변경허가: 10만원 (결혼이민 F-6: 4만원, 영주 F-5: 20만원)
    체류기간 연장허가: 6만원 (결혼이민 F-6: 3만원)
    단수재입국허가: 3만원
    복수재입국허가: 5만원
    외국인등록증/거소등록증 발급 및 재발급: 3만 5천원
    출입국/외국인등록/국내거소신고 사실증명: 2천원
- duration_or_validity:
- quota_or_limit:
- score_criteria:
- table_summary: 출입국관리법 제87조 등에 따른 체류 관련 수수료 일람
- table_rows:
- normalized_text:
- obligations:


## Stage 4: 정규화 MD ↔ 원본 MD 교차 검증

결정적 Python으로 각 정규화 행이 원본 청크에서 실제 등장하는지 검증합니다.
- 비자코드: 행이 주장하는 코드가 원본에 등장하는지
- 금액 (수수료): 원본에 있는지
- 서류명: 원본에 있는지 (lenient — 적어도 하나는 매치되어야 함)
- 필수 필드: 비어있는지

```bash
python scripts/validate_normalization.py
```


In [11]:
# 검증 결과 로드
def load_validation(manual_key: str):
    path = VAL / f"{manual_key}_validation.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

stay_v = load_validation("stay")
visa_v = load_validation("visa")

summary = pd.DataFrame([
    {"manual": "stay", "normalized_chunks": stay_v["normalized_chunks"],
     "total_rows": stay_v["total_rows"], "rows_with_issues": stay_v["rows_with_issues"]},
    {"manual": "visa", "normalized_chunks": visa_v["normalized_chunks"],
     "total_rows": visa_v["total_rows"], "rows_with_issues": visa_v["rows_with_issues"]},
])
summary["clean_rate"] = (1 - summary["rows_with_issues"] / summary["total_rows"]).round(3)
display(summary)


,manual,normalized_chunks,total_rows,rows_with_issues,clean_rate
0,stay,5,38,0,1.0
1,visa,1,6,0,1.0


In [12]:
# 이슈 종류별 빈도
issue_counts = {"stay": {}, "visa": {}}
for manual_key, v in [("stay", stay_v), ("visa", visa_v)]:
    for chunk in v["chunks"]:
        for row in chunk["rows"]:
            for issue in row.get("issues", []):
                kind = issue.split(":", 1)[0]
                issue_counts[manual_key][kind] = issue_counts[manual_key].get(kind, 0) + 1
        for issue in chunk.get("chunk_level_issues", []):
            kind = issue.split(":", 1)[0]
            issue_counts[manual_key][kind] = issue_counts[manual_key].get(kind, 0) + 1

if any(issue_counts["stay"].values()) or any(issue_counts["visa"].values()):
    issue_df = pd.DataFrame([
        {"manual": m, "issue": k, "count": c}
        for m, d in issue_counts.items() for k, c in d.items()
    ])
    fig = px.bar(issue_df, x="issue", y="count", color="manual", barmode="group",
                 title="Validator 이슈 종류별 빈도",
                 color_discrete_map={"stay": "#2563eb", "visa": "#7c3aed"})
    fig.show()
else:
    print("✓ 어떤 검증 이슈도 발생하지 않음 (clean) — LLM 추출이 원본과 모두 일치")


✓ 어떤 검증 이슈도 발생하지 않음 (clean) — LLM 추출이 원본과 모두 일치


## Stage 6: 정규화 MD → Semantic CSV

```bash
python scripts/build_semantic_csv.py
```

`data/parsed/normalized/*.md` → `data/processed/{stay,visa}_manual_semantic_clean.csv`

스키마 강제, 결정적 Python. CSV는 RAG 전처리의 입력으로 쓰입니다.


In [13]:
stay_semantic = pd.read_csv(PROC / "stay_manual_semantic_clean.csv")
visa_semantic = pd.read_csv(PROC / "visa_manual_semantic_clean.csv")
print(f"stay semantic: {stay_semantic.shape}")
print(f"visa semantic: {visa_semantic.shape}")
print(f"\nstay 컬럼 ({len(stay_semantic.columns)}):", list(stay_semantic.columns))
print(f"\nvisa 컬럼 ({len(visa_semantic.columns)}):", list(visa_semantic.columns))


stay semantic: (38, 27)
visa semantic: (6, 28)

stay 컬럼 (27): ['stay_status_code', 'stay_status_name_ko', 'manual_type', 'source_pdf', 'item_type', 'section_title', 'subtype_or_program', 'petition_type', 'subsection_type', 'applicant_context', 'eligibility', 'target_persons', 'common_documents', 'mandatory_documents', 'other_documents', 'requirements', 'procedure', 'restrictions', 'exceptions', 'fees', 'duration_or_validity', 'quota_or_limit', 'score_criteria', 'table_summary', 'table_rows', 'normalized_text', 'obligations']

visa 컬럼 (28): ['visa_code', 'visa_name_ko', 'manual_type', 'source_pdf', 'item_type', 'section_title', 'subtype_or_program', 'petition_type', 'subsection_type', 'applicant_context', 'eligibility', 'target_persons', 'common_documents', 'mandatory_documents', 'other_documents', 'requirements', 'procedure', 'restrictions', 'exceptions', 'fees', 'duration_or_validity', 'quota_or_limit', 'score_criteria', 'table_summary', 'table_rows', 'normalized_text', 'inviter_conte

In [14]:
# stay semantic CSV 미리보기 (주요 컬럼)
preview_cols = ["stay_status_code", "stay_status_name_ko", "item_type", "petition_type",
                "subsection_type", "section_title"]
display(stay_semantic[preview_cols].head(15))


,stay_status_code,stay_status_name_ko,item_type,petition_type,subsection_type,section_title
0,공통,공통사항,common_rule,공통사항,수수료,공통사항: 각종 체류허가 등에 관한 심사수수료
1,공통,공통사항,common_rule,공통사항,기간,공통사항: 여권 유효기간 범위 내 체류기간 부여
2,공통,공통사항,common_rule,공통사항,신고의무,공통사항: 직업 및 연간소득금액 신고 의무
3,공통,공통사항,common_rule,공통사항,신고의무,공통사항: 만 6세 이상 만 18세 이하 재학증명서 제출 의무
4,공통,공통사항,common_rule,공통사항,제출서류,공통사항: 외국인 결핵진단서 제출 의무
5,A-1,외교,stay_status_rule,공통사항,대상,A-1 외교 / 자격 해당자 및 활동범위
6,A-1,외교,stay_status_rule,체류자격외 활동허가,절차,A-1 외교 / 체류자격외 활동허가
7,A-1,외교,stay_status_rule,체류자격 부여,제출서류,A-1 외교 / 체류자격 부여 (출생자)
8,A-1,외교,stay_status_rule,체류자격 변경,제출서류,A-1 외교 / 체류자격 변경허가
9,A-1,외교,stay_status_rule,체류기간 연장,제출서류,A-1 외교 / 체류기간 연장허가


In [15]:
# 비자코드별 행 수 (stay)
code_dist = stay_semantic.groupby(["stay_status_code", "stay_status_name_ko"]).size().reset_index(name="rows")
code_dist["label"] = code_dist["stay_status_code"] + " " + code_dist["stay_status_name_ko"]
code_dist = code_dist.sort_values("rows", ascending=True)

fig = px.bar(code_dist, x="rows", y="label", orientation="h",
             title=f"Stay 매뉴얼: 비자코드별 추출 행 수 (총 {len(stay_semantic)})",
             color="rows", color_continuous_scale="Blues")
fig.update_layout(height=500, yaxis_title="", xaxis_title="rows")
fig.show()


In [16]:
# Petition type 분포 (stay)
pt_dist = stay_semantic.groupby("petition_type").size().reset_index(name="count")
pt_dist = pt_dist.sort_values("count", ascending=False)

fig = px.pie(pt_dist, values="count", names="petition_type",
             title="Stay: petition_type 분포",
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()


In [17]:
# Subsection_type 분포
sst_dist = stay_semantic.groupby(["petition_type", "subsection_type"]).size().reset_index(name="count")
fig = px.treemap(sst_dist, path=["petition_type", "subsection_type"], values="count",
                 title="Stay: petition_type × subsection_type 분포 (treemap)")
fig.update_traces(textinfo="label+value")
fig.show()


In [18]:
# 컬럼 누락률 히트맵
def fill_rate_heatmap(df, title):
    fill_rates = (df.notna() & (df.astype(str) != "")).mean().sort_values()
    fill_df = pd.DataFrame({"column": fill_rates.index, "fill_rate": fill_rates.values})
    fill_df = fill_df.sort_values("fill_rate", ascending=True)
    fig = px.bar(fill_df, x="fill_rate", y="column", orientation="h",
                 title=title,
                 color="fill_rate", color_continuous_scale="RdYlGn", range_color=[0, 1])
    fig.update_layout(height=700, xaxis_tickformat=".0%")
    return fig

fill_rate_heatmap(stay_semantic, "Stay semantic CSV — 컬럼별 채움 비율").show()


## Stage 7-8: Chatbot CSV (LLM 풍부화 + 결정적 빌드)

`/vizabridge-enrich-chatbot` 스킬이 semantic CSV의 각 행을 읽어, 사용자 자연어 상황 태그·키워드·라우팅 힌트를 생성합니다.
그 결과를 정규화 MD로 저장하고, `build_chatbot_csv.py`로 최종 CSV로 변환합니다.

```bash
# Claude Code 세션에서:
/vizabridge-enrich-chatbot stay
/vizabridge-enrich-chatbot visa

python scripts/build_chatbot_csv.py
```


In [19]:
stay_chatbot = pd.read_csv(PROC / "stay_manual_chatbot_ready.csv")
visa_chatbot = pd.read_csv(PROC / "visa_manual_chatbot_ready.csv")
print(f"stay chatbot: {stay_chatbot.shape}")
print(f"visa chatbot: {visa_chatbot.shape}")
print(f"\n컬럼:", list(stay_chatbot.columns))


stay chatbot: (22, 16)
visa chatbot: (6, 16)

컬럼: ['record_id', 'source_dataset', 'code_type', 'primary_code', 'primary_name_ko', 'source_section_title', 'user_situation_tags', 'intent_keywords', 'applicant_profile', 'current_location_context', 'current_status_context', 'plain_language_summary', 'required_user_info', 'routing_hint', 'answer_focus', 'search_text']


In [20]:
# 챗봇 CSV 핵심 컬럼 미리보기 (stay)
cb_cols = ["primary_code", "primary_name_ko", "user_situation_tags", "intent_keywords",
           "applicant_profile", "plain_language_summary"]
display(stay_chatbot[cb_cols].head(10))


,primary_code,primary_name_ko,user_situation_tags,intent_keywords,applicant_profile,plain_language_summary
0,공통,공통사항,- 공통사항,- 체류 신청 수수료 얼마\n- F-6 결혼이민 비자 수수료\n- 영주권 신청 비용\n- 체류 자격 변경 비용\n- 외국인등록증 발급 비용,외국인 본인 또는 대리인,체류자격 외 활동·근무처 변경·체류자격 부여/변경·기간연장·재입국허가·외국인등록증 발급 등 각종 체류 민원 수수료 안내
1,공통,공통사항,- 공통사항,- 여권 유효기간 짧은데 체류 가능한가\n- 여권 곧 만료되는데 체류기간 연장\n- 체류기간 부여 기준,장기체류자격 외국인,체류기간은 원칙적으로 여권 유효기간 범위 내 부여 (외교·공무·협정·영주·난민 제외)
2,공통,공통사항,- 취업/고용\n- 공통사항,- 외국인 직업 신고\n- 연간 소득금액 신고\n- 직업 변경 신고\n- 소득금액증명,취업 가능 체류자격 외국인,취업 가능 체류자격자는 직업과 연간 소득금액을 출입국·외국인관서에 신고해야 합니다
3,공통,공통사항,- 공통사항,- 외국인 자녀 학교 신고\n- 미성년 외국인 재학증명서\n- 초중고 외국인 학생 등록,만 6-18세 외국인 본인 또는 부모,만 6-18세 외국인은 초·중·고 재학 여부를 신고해야 합니다 (재학증명서 제출)
4,공통,공통사항,- 공통사항\n- 결핵검진,- 결핵진단서 어디서 받나\n- 베트남에서 한국 장기비자\n- 결핵 고위험국가 비자\n- 사증 신청 결핵검사 면제,결핵 고위험국가 35개국 국민,결핵 고위험국가 35개국 국민은 90일 초과 사증 신청·외국인등록·체류허가 시 결핵진단서 제출
5,A-1,외교,- 공무/외교,- 외교관 가족 한국 체류\n- 외교 비자 A-1 대상자\n- 국제기구 직원 한국,외교관 또는 외교사절단 가족,"외교사절단·영사기관 구성원과 그 가족, 그리고 외교사절과 동등한 특권 면제를 받는 자에게 부여되는 체류자격"
6,A-1,외교,- 공무/외교\n- 취업/고용\n- 자격외활동,- 외교관 가족 한국 취업\n- 주한외국공관 가족 일자리\n- 외교부 고용추천서\n- 미국 대사관 가족 취업,주한외국공관원 가족 또는 국제기구 직원 동반가족,주한외국공관원 가족과 국제기구 직원 동반가족이 한국에서 취업하려면 외교부 고용추천서를 받아 체류자격외 활동허가를 받습니다
7,A-1,외교,- 공무/외교\n- 가족초청/동반,- 외교관 자녀 출생 체류자격\n- 외교 자격 신규 부여\n- 출생일 90일 이내 신청,외교관 부모가 한국 내 출생 자녀,외교관 가족 자녀가 한국에서 출생한 경우 출생일부터 90일 이내 체류자격 부여 신청
8,A-1,외교,- 공무/외교\n- 자격변경,- 외교 자격으로 변경\n- 외교사절 가족 자격변경\n- 동반가족 A-1 변경,외교사절·영사기관 구성원 또는 그 동반가족,외교사절단·영사기관 구성원과 동반가족이 다른 자격으로 입국 후 외교(A-1) 자격으로 변경
9,A-1,외교,- 공무/외교\n- 기간연장,- 외교 체류기간 연장\n- 재임기간 내 연장,외교 자격 소지자,재임기간 범위 내에서 체류기간 연장 (수수료 면제)


In [21]:
# 첫 번째 챗봇 행의 모든 필드 풀 디스플레이 (rich)
sample = stay_chatbot.iloc[0]
print("=" * 70)
print(f"record_id: {sample['record_id']}")
print(f"primary: {sample['primary_code']} {sample['primary_name_ko']}")
print("=" * 70)
for col in stay_chatbot.columns:
    v = str(sample[col]).strip() if pd.notna(sample[col]) else ""
    if v:
        print(f"\n[{col}]")
        print(v[:500] + ("..." if len(v) > 500 else ""))


record_id: stay-00000-공통
primary: 공통 공통사항

[record_id]
stay-00000-공통

[source_dataset]
stay_manual_semantic_clean.csv

[code_type]
stay_status

[primary_code]
공통

[primary_name_ko]
공통사항

[source_section_title]
공통사항: 각종 체류허가 등에 관한 심사수수료

[user_situation_tags]
- 공통사항

[intent_keywords]
- 체류 신청 수수료 얼마
- F-6 결혼이민 비자 수수료
- 영주권 신청 비용
- 체류 자격 변경 비용
- 외국인등록증 발급 비용

[applicant_profile]
외국인 본인 또는 대리인

[current_location_context]
국내 체류 중 민원

[current_status_context]
장기체류자격 외국인

[plain_language_summary]
체류자격 외 활동·근무처 변경·체류자격 부여/변경·기간연장·재입국허가·외국인등록증 발급 등 각종 체류 민원 수수료 안내

[required_user_info]
- 신청하려는 민원 유형
- 결혼이민(F-6) 또는 영주(F-5) 여부

[routing_hint]
체류민원 / 공통사항

[answer_focus]
수수료

[search_text]
체류 수수료 12만원 10만원 6만원 8만원 3만5천원 외국인등록증 체류기간 연장 자격변경 결혼이민 영주


In [22]:
# 상황 태그 빈도 (가장 흔한 user_situation_tags)
def parse_tags(s):
    if pd.isna(s): return []
    return [line.strip().lstrip("- ").strip() for line in str(s).splitlines() if line.strip() and line.strip() != "-"]

stay_chatbot["tag_list"] = stay_chatbot["user_situation_tags"].apply(parse_tags)
all_tags = [t for tags in stay_chatbot["tag_list"] for t in tags if t]
tag_df = pd.Series(all_tags).value_counts().reset_index()
tag_df.columns = ["tag", "count"]

fig = px.bar(tag_df, x="count", y="tag", orientation="h",
             title=f"Stay chatbot: 상황 태그 빈도 (총 {len(stay_chatbot)} 행)",
             color="count", color_continuous_scale="Viridis")
fig.update_layout(height=500)
fig.show()


In [23]:
# Intent keywords 워드 카운트 (간단)
def parse_keywords(s):
    if pd.isna(s): return []
    return [line.strip().lstrip("- ").strip() for line in str(s).splitlines()
            if line.strip() and not line.strip().startswith("-") == False]

stay_chatbot["kw_list"] = stay_chatbot["intent_keywords"].apply(parse_keywords)
total_keywords = sum(len(kws) for kws in stay_chatbot["kw_list"])
avg_keywords = total_keywords / len(stay_chatbot)
print(f"총 키워드 phrase: {total_keywords}")
print(f"행 평균: {avg_keywords:.1f}")
print()
print("샘플 키워드 (stay row 4, 결핵진단서):")
for kw in stay_chatbot.iloc[4]["kw_list"][:6]:
    print(f"  - {kw}")


총 키워드 phrase: 76
행 평균: 3.5

샘플 키워드 (stay row 4, 결핵진단서):
  - 결핵진단서 어디서 받나
  - 베트남에서 한국 장기비자
  - 결핵 고위험국가 비자
  - 사증 신청 결핵검사 면제


In [24]:
# 라우팅 힌트별 행 수
route_df = stay_chatbot.groupby("routing_hint").size().reset_index(name="count")
route_df = route_df.sort_values("count", ascending=True)

fig = px.bar(route_df, x="count", y="routing_hint", orientation="h",
             title="Stay chatbot: routing_hint 별 행 수 (어떤 민원유형으로 라우팅되는가)",
             color="count", color_continuous_scale="Plasma")
fig.update_layout(height=400)
fig.show()


## Stage 9: 품질 리포트

`scripts/quality_report_semantic_manual_csvs.py`가 검수용 산출물을 생성합니다.

- `output/quality/*_manual_quality_summary.csv` — 요약 통계
- `output/quality/*_manual_field_fill_rates.csv` — 컬럼별 채움률
- `output/quality/*_manual_review_candidates.csv` — 검수 후보 행
- `output/review/*_manual_review.xlsx` — 사람이 보는 Excel


In [25]:
# 품질 요약 (stay)
stay_qsum = pd.read_csv(QUAL / "stay_manual_quality_summary.csv")
display(stay_qsum)


,metric,value
0,manual_key,stay
1,manual_label,체류민원
2,row_count,38
3,column_count,27
4,review_candidate_count,26
5,review_candidate_rate,68.42%
6,high_priority_count,26
7,medium_priority_count,0
8,low_priority_count,0
9,lowest_fill_rate_fields,common_documents:0%; normalized_text:0%; other_documents:0%; quota_or_limit:...


In [26]:
# Field fill rates
stay_fr = pd.read_csv(QUAL / "stay_manual_field_fill_rates.csv")
stay_fr = stay_fr.sort_values("fill_rate", ascending=True)
fig = px.bar(stay_fr, x="fill_rate", y="field", orientation="h",
             title="Stay semantic CSV — quality_report 산출 컬럼별 채움률",
             color="fill_rate", color_continuous_scale="RdYlGn", range_color=[0, 1])
fig.update_layout(height=700, xaxis_tickformat=".0%")
fig.show()


In [27]:
# 검수 후보 (stay)
stay_cand = pd.read_csv(QUAL / "stay_manual_review_candidates.csv")
print(f"stay 검수 후보 행: {len(stay_cand)}")
if len(stay_cand) > 0:
    display(stay_cand[["section_title", "petition_type", "subsection_type",
                       "review_priority", "review_reason"]].head(10))
else:
    print("\n→ 검수 후보 0건. 모든 행이 임계값 통과.")


stay 검수 후보 행: 26


,section_title,petition_type,subsection_type,review_priority,review_reason
0,공통사항: 여권 유효기간 범위 내 체류기간 부여,공통사항,기간,high,document_like_row_outside_required_documents
1,공통사항: 직업 및 연간소득금액 신고 의무,공통사항,신고의무,high,document_like_row_outside_required_documents
2,공통사항: 만 6세 이상 만 18세 이하 재학증명서 제출 의무,공통사항,신고의무,high,document_like_row_outside_required_documents
3,공통사항: 외국인 결핵진단서 제출 의무,공통사항,제출서류,high,document_like_row_outside_required_documents
4,A-1 외교 / 체류자격외 활동허가,체류자격외 활동허가,절차,high,document_like_row_outside_required_documents; score_like_row_outside_score_t...
5,A-1 외교 / 체류자격 부여 (출생자),체류자격 부여,제출서류,high,document_like_row_outside_required_documents
6,A-1 외교 / 체류자격 변경허가,체류자격 변경,제출서류,high,document_like_row_outside_required_documents
7,A-1 외교 / 체류기간 연장허가,체류기간 연장,제출서류,high,document_like_row_outside_required_documents
8,A-1 외교 / 재입국허가,재입국허가,제출서류,high,document_like_row_outside_required_documents
9,A-1 외교 / 외국인등록,외국인등록,제출서류,high,document_like_row_outside_required_documents


## End-to-End 요약

각 단계 산출물과 행 수의 흐름을 한눈에 봅니다.


In [28]:
# 전체 파이프라인 통계
stats = {
    "Stage 1 (HWP→MD)": {
        "stay": (RAW / "stay_manual.md").stat().st_size // 1024,
        "visa": (RAW / "visa_manual.md").stat().st_size // 1024,
        "unit": "KB",
    },
    "Stage 2 (chunks)": {
        "stay": len(stay_chunks),
        "visa": len(visa_chunks),
        "unit": "chunks",
    },
    "Stage 3 (normalized rows)": {
        "stay": stay_p["total_rows"],
        "visa": visa_p["total_rows"],
        "unit": "rows",
    },
    "Stage 4 (validation issues)": {
        "stay": stay_v["rows_with_issues"],
        "visa": visa_v["rows_with_issues"],
        "unit": "issues",
    },
    "Stage 6 (semantic CSV)": {
        "stay": len(stay_semantic),
        "visa": len(visa_semantic),
        "unit": "rows",
    },
    "Stage 8 (chatbot CSV)": {
        "stay": len(stay_chatbot),
        "visa": len(visa_chatbot),
        "unit": "rows",
    },
}
print(f"{'Stage':<35} {'Stay':>10} {'Visa':>10} {'Unit':>10}")
print("─" * 70)
for stage, data in stats.items():
    print(f"{stage:<35} {data['stay']:>10} {data['visa']:>10} {data['unit']:>10}")


Stage                                     Stay       Visa       Unit
──────────────────────────────────────────────────────────────────────
Stage 1 (HWP→MD)                          1612       1059         KB
Stage 2 (chunks)                            34         23     chunks
Stage 3 (normalized rows)                   38          6       rows
Stage 4 (validation issues)                  0          0     issues
Stage 6 (semantic CSV)                      38          6       rows
Stage 8 (chatbot CSV)                       22          6       rows


In [29]:
# 비자코드 흐름: chunks에서 발견된 코드 vs semantic CSV에 등장한 코드
stay_chunks_codes = set().union(*stay_chunks["visa_codes"].tolist())
stay_semantic_codes = set(stay_semantic["stay_status_code"].dropna().unique()) - {"공통", ""}
stay_chatbot_codes = set(stay_chatbot["primary_code"].dropna().unique()) - {"공통", ""}

stage_codes = pd.DataFrame({
    "stage": ["chunks (발견)", "semantic CSV", "chatbot CSV"],
    "unique_codes": [len(stay_chunks_codes), len(stay_semantic_codes), len(stay_chatbot_codes)],
})
display(stage_codes)
print()
print(f"chunks에서만 발견 (semantic CSV에 미반영): {sorted(stay_chunks_codes - stay_semantic_codes)[:20]}")
print(f"semantic CSV의 코드: {sorted(stay_semantic_codes)}")


,stage,unique_codes
0,chunks (발견),181
1,semantic CSV,12
2,chatbot CSV,8



chunks에서만 발견 (semantic CSV에 미반영): ['A-3-99', 'C-2', 'C-3-1', 'C-3-10', 'C-3-2', 'C-3-3', 'C-3-4', 'C-3-5', 'C-3-6', 'C-3-7', 'C-3-8', 'C-3-9', 'C-4-1', 'C-4-5', 'D-10', 'D-10-1', 'D-10-2', 'D-10-3', 'D-10-T', 'D-2-1']
semantic CSV의 코드: ['A-1', 'A-2', 'A-3', 'B-1', 'B-2', 'C-1', 'C-3', 'C-4', 'D-1', 'D-2', 'D-3', 'D-4']


In [30]:
# 최종 산출물 디렉토리 리스팅
print("=== Final outputs ===\n")
for d, files in [
    (PROC, "data/processed/*.csv"),
    (NORM, "data/parsed/normalized/*.md"),
    (NORM_CB, "data/parsed/normalized_chatbot/*.md"),
    (CHUNKS, "data/parsed/chunks/*.jsonl"),
    (VAL, "data/parsed/validation/*.json"),
    (QUAL, "output/quality/*"),
    (REV, "output/review/*.xlsx"),
]:
    print(f"## {files}")
    if d.exists():
        for p in sorted(d.iterdir()):
            if p.is_file() and not p.name.startswith("."):
                size_kb = p.stat().st_size / 1024
                print(f"  {p.name}: {size_kb:.1f} KB")
    print()


=== Final outputs ===

## data/processed/*.csv
  stay_manual_chatbot_ready.csv: 16.9 KB
  stay_manual_semantic_clean.csv: 28.0 KB
  visa_manual_chatbot_ready.csv: 4.3 KB
  visa_manual_semantic_clean.csv: 4.6 KB

## data/parsed/normalized/*.md
  stay_manual.md: 38.2 KB
  visa_manual.md: 5.2 KB

## data/parsed/normalized_chatbot/*.md
  stay_manual.md: 27.4 KB
  visa_manual.md: 6.9 KB

## data/parsed/chunks/*.jsonl
  stay_chunks_index.jsonl: 12.3 KB
  visa_chunks_index.jsonl: 7.6 KB

## data/parsed/validation/*.json
  stay_validation.json: 9.8 KB
  visa_validation.json: 1.6 KB

## output/quality/*
  semantic_manual_quality_report.md: 0.7 KB
  stay_manual_field_fill_rates.csv: 0.7 KB
  stay_manual_quality_summary.csv: 0.7 KB
  stay_manual_review_candidates.csv: 21.3 KB
  visa_manual_field_fill_rates.csv: 0.7 KB
  visa_manual_quality_summary.csv: 0.8 KB
  visa_manual_review_candidates.csv: 2.8 KB

## output/review/*.xlsx
  stay_manual_review.xlsx: 34.4 KB
  visa_manual_review.xlsx: 15.2 KB


## 결론

이 노트북은 9단계 파이프라인을 처음부터 끝까지 실행한 결과를 보여줍니다.

### 검증된 핵심 속성
1. **HWP → kordoc → Markdown 변환이 한글과 표 구조를 보존**
2. **결정적 청크 분할**이 모든 비자코드(sub-code 포함)를 발견
3. **LLM 기반 정규화 + 결정적 검증**이 할루시네이션 없이 의미 단위 행 생성
4. **정규화 MD가 깃에 커밋된 자산**으로, CSV는 언제든 결정적으로 재생성 가능
5. **챗봇 풍부화 단계**가 자연어 사용자 상황을 비자코드로 라우팅하는 데이터 추가
6. **결정적 품질 리포트**가 검수 후보를 식별

### 이 노트북에 들어간 데이터는 demonstration scope
- 전체 매뉴얼이 아닌, 핵심 비자코드 25-30개를 커버하는 대표 행 추출
- 실제 운영에서는 stays 34청크 × visa 23청크 전체를 normalize 스킬로 처리하면 ~200-300 semantic rows + 동일 enrichment 가능

### 다음 단계 (운영)
1. 남은 청크 추가 정규화: Claude Code 세션에서 `/vizabridge-normalize stay` 또는 `visa`
2. 추가 enrich: `/vizabridge-enrich-chatbot stay` 또는 `visa`
3. 매뉴얼 갱신 시: `data/raw/`의 HWP만 교체 후 stage 1부터 다시
